In [ ]:
# ============================================================
# Experiment 1 – Eye-Tracking Preprocessing
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# Parameters
# ============================================================

TFF_MIN = 100
TFF_TRIM_SD = 2

participants_to_exclude = [33, 35, 37, 54, 57, 58]

PARTICIPANTS_80_TRIALS = [1, 2, 3, 61, 62, 63, 64, 65]

# ============================================================
# Helper Functions
# ============================================================

def load_csv(name):
    return pd.read_csv(name, header=None).to_numpy(dtype=float)

def merge_sessions(arr):

    merged = []

    for p in range(arr.shape[1] // 2):

        merged.append(
            np.concatenate([
                arr[:, 2*p],
                arr[:, 2*p+1]
            ])
        )

    return merged

def trim_sd(x, sd=2):

    out = x.copy()

    m = np.nanmean(out)
    s = np.nanstd(out, ddof=1)

    lo = m - sd * s
    hi = m + sd * s

    out[(out < lo) | (out > hi)] = np.nan

    return out

# ============================================================
# Load Data
# ============================================================

tff_raw = load_csv("Raw_Data/timeFirstFixation.csv")

# ============================================================
# Merge Sessions
# ============================================================

tff_parts = merge_sessions(tff_raw)

# ============================================================
# Preprocess TFF
# ============================================================

rows = []

for pid in range(1, len(tff_parts) + 1):

    if pid in participants_to_exclude:
        continue

    tff = tff_parts[pid - 1]

    # Remove TFF < 100 ms
    tff[tff < TFF_MIN] = np.nan

    # Session-wise trimming
    s1 = trim_sd(tff[:48], sd=TFF_TRIM_SD)
    s2 = trim_sd(tff[48:], sd=TFF_TRIM_SD)

    tff_clean = np.concatenate([s1, s2])

    valid = np.isfinite(tff_clean)

    trial_counter = 1

    for t in np.where(valid)[0]:

        rows.append({

            "participant": pid,

            "trial": trial_counter,

            "TFF_ms": float(tff_clean[t]),

            "logTFF": float(np.log10(tff_clean[t]))
        })

        trial_counter += 1

# ============================================================
# Save
# ============================================================

df_et = pd.DataFrame(rows)

df_et.to_csv(
    "Processed_Data/Exp1_ET_clean.csv",
    index=False
)

print("Eye-tracking preprocessing complete.")

In [ ]:
# ============================================================
# Create Final Analysis Dataset
# ============================================================

import pandas as pd

# ============================================================
# Load Cleaned Data
# ============================================================

beh = pd.read_csv(
    "Processed_Data/Exp1_behavioural_clean.csv"
)

et = pd.read_csv(
    "Processed_Data/Exp1_ET_clean.csv"
)

# ============================================================
# Merge
# ============================================================

df = beh.merge(
    et,
    on=["participant", "trial"],
    how="inner"
)

# ============================================================
# Effect Coding
# ============================================================

df["side"] = df["side"].map({
    "left": -1,
    "right": 1
})

df["valence"] = df["valence"].map({
    "negative": -1,
    "positive": 1
})

# ============================================================
# Trial Index
# ============================================================

df["trial_index"] = (
    df.groupby("participant")
      .cumcount() + 1
)

# ============================================================
# Save
# ============================================================

df.to_csv(
    "Processed_Data/Exp1_ready.csv",
    index=False
)

print("Final analysis dataset saved.")